# Notebook 03: Gold Aggregations (Business Analytics & Dashboard Serving)

Merges player on-pitch performance (FPL stats) with fan sentiment (Reddit discussions) across three analytical granularities:
1. **Monthly Summary**: `performance_vs_toxicity.gold.player_month_summary`
2. **Matchweek Summary**: `performance_vs_toxicity.gold.player_gameweek_summary` (enriched with fixture details, opponents, home/away)
3. **Daily Sentiment**: `performance_vs_toxicity.gold.player_daily_sentiment`

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is available on sys.path dynamically
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.append(repo_root)

import pandas as pd
from src.common.config import load_config
from src.ingestion.fetch_fpl_season import _download_csv
from src.aggregation.merge_performance_sentiment import (
    aggregate_performance, aggregate_sentiment, build_gold_dataset,
    aggregate_performance_by_gameweek, aggregate_sentiment_by_gameweek, build_gold_dataset_by_gameweek,
    aggregate_sentiment_by_day,
)

cfg = load_config()
RAW_DIR = "/Volumes/performance_vs_toxicity/bronze/raw_files"
tagged_comments = spark.table("performance_vs_toxicity.silver.tagged_comments")

month_frames, gw_frames, day_frames = [], [], []

for season_id in cfg["seasons"]:
    # Load raw FPL matchweeks and teams reference
    gameweeks_df = pd.read_csv(f"{RAW_DIR}/fpl/{season_id}/gameweeks.csv")
    fpl_dir = cfg["seasons"][season_id]["fpl_season_dir"]
    teams_df = _download_csv(f"{cfg['fpl']['base_url']}/{fpl_dir}/teams.csv")
    
    # Filter silver tagged comments for the current season
    tagged_pd = tagged_comments.filter(f"season = '{season_id}'").toPandas()

    # 1. Monthly Granularity
    perf_month = aggregate_performance(gameweeks_df)
    if len(tagged_pd):
        sent_month = aggregate_sentiment(tagged_pd)
    else:
        sent_month = pd.DataFrame(columns=["player_id", "month", "avg_sentiment", "n_comments", "negative_share"])
    gold_month = build_gold_dataset(perf_month, sent_month)
    gold_month["season"] = season_id
    month_frames.append(gold_month)

    # 2. Matchweek Granularity
    perf_gw = aggregate_performance_by_gameweek(gameweeks_df, teams_df)
    if len(tagged_pd):
        sent_gw = aggregate_sentiment_by_gameweek(tagged_pd, gameweeks_df)
    else:
        sent_gw = pd.DataFrame(columns=["player_id", "player_name", "round", "avg_sentiment", "n_comments", "negative_share"])
    gold_gw = build_gold_dataset_by_gameweek(perf_gw, sent_gw)
    gold_gw["season"] = season_id
    gw_frames.append(gold_gw)

    # 3. Daily Granularity
    if len(tagged_pd):
        sent_day = aggregate_sentiment_by_day(tagged_pd)
    else:
        sent_day = pd.DataFrame(columns=["player_id", "player_name", "date", "avg_sentiment", "n_comments", "negative_share"])
    sent_day["season"] = season_id
    day_frames.append(sent_day)

# Write curated Gold tables to Delta Lake
spark.createDataFrame(pd.concat(month_frames, ignore_index=True)).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("performance_vs_toxicity.gold.player_month_summary")

spark.createDataFrame(pd.concat(gw_frames, ignore_index=True)).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("performance_vs_toxicity.gold.player_gameweek_summary")

spark.createDataFrame(pd.concat(day_frames, ignore_index=True)).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("performance_vs_toxicity.gold.player_daily_sentiment")

print("Successfully created Gold Delta tables: player_month_summary, player_gameweek_summary, player_daily_sentiment")